# Prepare clinical MRI cohort for direct Jacobian vs Wasserstein

Build a lean, session-level clinical cohort. MRI sessions are matched to the nearest diagnosis and CDR visits from the same participant.

In [34]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / 'data' / 'database_finale_labels_corrette.csv').is_file()
)

BASE_PATH = PROJECT_ROOT / 'data' / 'database_finale_labels_corrette.csv'
DIAGNOSIS_PATH = PROJECT_ROOT / 'csv_source_oasis' / 'OASIS3_UDSd1_diagnoses.csv'
CDR_PATH = PROJECT_ROOT / 'csv_source_oasis' / 'OASIS3_UDSb4_cdr.csv'
RUN_ROOT = PROJECT_ROOT / 'outputs' / 'cohorts' / 'oasis3' / 'runs' / 'connected-atlas-8cf3dc8f'
DIRECT_ROOT = RUN_ROOT / 'parcel_vectors'
WEIGHTED_DEGREE_ROOT = RUN_ROOT / 'graphs' / 'unthresholded' / 'was' / 'expW'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'derived' / 'oasis3_clinical_mri_cohort.csv'


MAX_CLINICAL_GAP_DAYS = 180

In [35]:
# 1. Keep MRI-session identifiers, covariates, and the T1 path.
base_columns = [
    'OASISID', 'Enr-Day', 'T1w Scan Path', 'age at visit', 'GENDER', 'EDUC'
]
mri_sessions = pd.read_csv(BASE_PATH, dtype={'Enr-Day': 'string'})[base_columns].copy()
mri_sessions['mri_day'] = (
    mri_sessions['Enr-Day'].str.extract(r'(\d+)')[0].astype('Int64')
)
mri_sessions['mri_row'] = np.arange(len(mri_sessions))
mri_sessions.head()

,OASISID,Enr-Day,T1w Scan Path,age at visit,GENDER,EDUC,mri_day,mri_row
0,OAS30001,d0129,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,65.19,Female,12.0,129,0
1,OAS30002,d0371,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,67.25,Male,18.0,371,1
2,OAS30003,d0558,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,58.81,Female,18.0,558,2
3,OAS30004,d1101,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,55.13,Female,17.0,1101,3
4,OAS30005,d0143,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,48.06,Female,16.0,143,4


In [36]:
# 2. Retain only diagnosis columns needed to define a clinical group.
diagnosis_columns = [
    'OASISID', 'days_to_visit', 'NORMCOG', 'DEMENTED',
    'PROBAD', 'PROBADIF', 'POSSAD', 'alzdis', 'alzdisif'
]
diagnosis_visits = pd.read_csv(DIAGNOSIS_PATH, usecols=diagnosis_columns).copy()
diagnosis_visits['d1_visit_day'] = pd.to_numeric(
    diagnosis_visits.pop('days_to_visit'), errors='coerce'
)
diagnosis_visits = diagnosis_visits.rename(columns={
    column: f'd1_{column.lower()}'
    for column in diagnosis_visits.columns
    if column not in {'OASISID', 'd1_visit_day'}
})
diagnosis_visits.head()

,OASISID,d1_normcog,d1_demented,d1_probad,d1_probadif,d1_possad,d1_alzdis,d1_alzdisif,d1_visit_day
0,OAS30001,1.0,NaN,NaN,NaN,NaN,NaN,NaN,0
1,OAS30001,1.0,NaN,NaN,NaN,NaN,NaN,NaN,339
2,OAS30001,1.0,NaN,NaN,NaN,NaN,NaN,NaN,722
3,OAS30001,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1106
4,OAS30001,1.0,NaN,NaN,NaN,NaN,NaN,NaN,1456


In [37]:
# 3. Retain CDR severity and cognitive-status columns.
cdr_columns = ['OASISID', 'days_to_visit', 'CDRTOT', 'CDRSUM', 'MMSE', 'dx1']
cdr_visits = pd.read_csv(CDR_PATH, usecols=cdr_columns).copy()
cdr_visits['cdr_visit_day'] = pd.to_numeric(
    cdr_visits.pop('days_to_visit'), errors='coerce'
)
cdr_visits = cdr_visits.rename(columns={
    'CDRTOT': 'cdr_total',
    'CDRSUM': 'cdr_sum',
    'MMSE': 'cdr_mmse',
    'dx1': 'cdr_dx1',
})
cdr_visits.head()

,OASISID,cdr_mmse,cdr_sum,cdr_total,cdr_dx1,cdr_visit_day
0,OAS30001,28.0,0.0,0.0,Cognitively normal,0
1,OAS30001,28.0,0.0,0.0,Cognitively normal,339
2,OAS30001,30.0,0.0,0.0,Cognitively normal,722
3,OAS30001,30.0,0.0,0.0,Cognitively normal,1106
4,OAS30001,30.0,0.0,0.0,Cognitively normal,1456


In [38]:
# 4. Match each MRI to the nearest visit of one clinical table, within the same OASISID.
def nearest_visit_match(mri, visits, visit_day_column, prefix):
    matches = []
    value_columns = [column for column in visits.columns if column != 'OASISID']

    for oasis_id, scans in mri.groupby('OASISID', sort=False):
        left = (
            scans[['mri_row', 'mri_day']].dropna()
            .assign(mri_day=lambda frame: frame['mri_day'].astype('int64'))
            .sort_values('mri_day')
        )
        right = visits.loc[visits['OASISID'].eq(oasis_id), value_columns].dropna(
            subset=[visit_day_column]
        ).assign(**{visit_day_column: lambda frame: frame[visit_day_column].astype('int64')})
        right = right.sort_values(visit_day_column)

        if right.empty:
            empty = left.copy()
            for column in value_columns:
                empty[column] = np.nan
            matches.append(empty)
        else:
            matches.append(pd.merge_asof(
                left, right, left_on='mri_day', right_on=visit_day_column, direction='nearest'
            ))

    matched = pd.concat(matches, ignore_index=True)
    matched[f'{prefix}_gap_days_signed'] = matched[visit_day_column] - matched['mri_day']
    matched[f'{prefix}_gap_days_abs'] = matched[f'{prefix}_gap_days_signed'].abs()
    return matched.drop(columns='mri_day')

d1_matches = nearest_visit_match(mri_sessions, diagnosis_visits, 'd1_visit_day', 'd1')
cdr_matches = nearest_visit_match(mri_sessions, cdr_visits, 'cdr_visit_day', 'cdr')

clinical_mri = (
    mri_sessions
    .merge(d1_matches, on='mri_row', how='left', validate='one_to_one')
    .merge(cdr_matches, on='mri_row', how='left', validate='one_to_one')
)
clinical_mri[[
    'OASISID', 'Enr-Day', 'mri_day', 'd1_visit_day', 'd1_gap_days_signed',
    'cdr_visit_day', 'cdr_gap_days_signed', 'cdr_total'
]].head()

,OASISID,Enr-Day,mri_day,d1_visit_day,d1_gap_days_signed,cdr_visit_day,cdr_gap_days_signed,cdr_total
0,OAS30001,d0129,129,0.0,-129.0,0,-129,0.0
1,OAS30002,d0371,371,0.0,-371.0,0,-371,0.0
2,OAS30003,d0558,558,1083.0,525.0,1083,525,0.0
3,OAS30004,d1101,1101,1102.0,1.0,1102,1,0.0
4,OAS30005,d0143,143,0.0,-143.0,0,-143,0.0


In [39]:
# 5. Apply the time window, then create conservative clinical labels.
clinical_mri['d1_within_window'] = clinical_mri['d1_gap_days_abs'].le(MAX_CLINICAL_GAP_DAYS)
clinical_mri['cdr_within_window'] = clinical_mri['cdr_gap_days_abs'].le(MAX_CLINICAL_GAP_DAYS)

primary_ad = (
    clinical_mri['d1_demented'].eq(1)
    & (
        (clinical_mri['d1_alzdis'].eq(1) & clinical_mri['d1_alzdisif'].eq(1))
        | (clinical_mri['d1_probad'].eq(1) & clinical_mri['d1_probadif'].eq(1))
    )
)
clinically_normal = clinical_mri['d1_normcog'].eq(1) & clinical_mri['cdr_total'].eq(0)

clinical_mri['clinical_group'] = np.select(
    [
        primary_ad & clinical_mri['d1_within_window'] & clinical_mri['cdr_within_window'],
        clinically_normal & clinical_mri['d1_within_window'] & clinical_mri['cdr_within_window'],
    ],
    ['primary_ad_dementia', 'clinically_normal'],
    default=pd.NA,
)
clinical_mri['clinical_label_ad'] = clinical_mri['clinical_group'].map({
    'clinically_normal': 0,
    'primary_ad_dementia': 1,
}).astype('Int64')

clinical_mri['clinical_group'].value_counts(dropna=False)

clinical_group
clinically_normal      793
<NA>                   388
primary_ad_dementia    195
Name: count, dtype: int64

In [40]:
# 6. Record whether both precomputed imaging feature vectors are available.
clinical_mri['subject_id'] = (
    'sub-' + clinical_mri['OASISID'].str.removeprefix('OAS3')
)
clinical_mri['direct_mean_path'] = clinical_mri['subject_id'].map(
    lambda subject: str(DIRECT_ROOT / subject / 'direct_mean.dat')
)
clinical_mri['weighted_degree_path'] = clinical_mri['subject_id'].map(
    lambda subject: str(WEIGHTED_DEGREE_ROOT / subject / 'weighted_degree.dat')
)
clinical_mri['has_direct_mean'] = clinical_mri['direct_mean_path'].map(
    lambda path: Path(path).is_file()
)
clinical_mri['has_weighted_degree'] = clinical_mri['weighted_degree_path'].map(
    lambda path: Path(path).is_file()
)

model_cohort = clinical_mri.loc[
    clinical_mri['clinical_group'].notna()
    & clinical_mri['has_direct_mean']
    & clinical_mri['has_weighted_degree']
].copy()

model_cohort.groupby('clinical_group').agg(
    participants=('OASISID', 'nunique'),
    mri_sessions=('OASISID', 'size'),
    median_d1_gap_days=('d1_gap_days_abs', 'median'),
    median_cdr_gap_days=('cdr_gap_days_abs', 'median'),
)

,participants,mri_sessions,median_d1_gap_days,median_cdr_gap_days
clinical_group,,,,
clinically_normal,772,772,73.5,74.0
primary_ad_dementia,190,190,77.5,77.5


In [41]:
# 7. Save the lean modelling table. No HStatus-derived label is retained.
output_columns = [
    'OASISID', 'subject_id', 'Enr-Day', 'mri_day', 'T1w Scan Path',
    'age at visit', 'GENDER', 'EDUC',
    'd1_visit_day', 'd1_gap_days_signed', 'd1_gap_days_abs',
    'cdr_visit_day', 'cdr_gap_days_signed', 'cdr_gap_days_abs',
    'd1_normcog', 'd1_demented', 'd1_probad', 'd1_probadif',
    'd1_alzdis', 'd1_alzdisif', 'cdr_total', 'cdr_sum', 'cdr_mmse', 'cdr_dx1',
    'clinical_group', 'clinical_label_ad',
    'direct_mean_path', 'weighted_degree_path',
]

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
model_cohort[output_columns].sort_values(['clinical_label_ad', 'OASISID']).to_csv(
    OUTPUT_PATH, index=False
)

print(f'Saved: {OUTPUT_PATH}')
model_cohort[output_columns].head()

Saved: /home/lucagalli/Projects/parcellating_dbm/data/derived/oasis3_clinical_mri_cohort.csv


,OASISID,subject_id,Enr-Day,mri_day,T1w Scan Path,age at visit,GENDER,EDUC,d1_visit_day,d1_gap_days_signed,...,d1_alzdis,d1_alzdisif,cdr_total,cdr_sum,cdr_mmse,cdr_dx1,clinical_group,clinical_label_ad,direct_mean_path,weighted_degree_path
0,OAS30001,sub-0001,d0129,129,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,65.19,Female,12.0,0.0,-129.0,...,NaN,NaN,0.0,0.0,28.0,Cognitively normal,clinically_normal,0,/home/lucagalli/Projects/parcellating_dbm/outp...,/home/lucagalli/Projects/parcellating_dbm/outp...
3,OAS30004,sub-0004,d1101,1101,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,55.13,Female,17.0,1102.0,1.0,...,NaN,NaN,0.0,0.0,29.0,Cognitively normal,clinically_normal,0,/home/lucagalli/Projects/parcellating_dbm/outp...,/home/lucagalli/Projects/parcellating_dbm/outp...
4,OAS30005,sub-0005,d0143,143,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,48.06,Female,16.0,0.0,-143.0,...,NaN,NaN,0.0,0.0,29.0,Cognitively normal,clinically_normal,0,/home/lucagalli/Projects/parcellating_dbm/outp...,/home/lucagalli/Projects/parcellating_dbm/outp...
5,OAS30006,sub-0006,d0166,166,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,62.13,Male,16.0,0.0,-166.0,...,NaN,NaN,0.0,0.0,27.0,Cognitively normal,clinically_normal,0,/home/lucagalli/Projects/parcellating_dbm/outp...,/home/lucagalli/Projects/parcellating_dbm/outp...
6,OAS30007,sub-0007,d0061,61,/home/lucagalli/Projects/OASIS3_Pipeline_Luca/...,71.59,Male,18.0,0.0,-61.0,...,NaN,NaN,0.0,0.0,29.0,Cognitively normal,clinically_normal,0,/home/lucagalli/Projects/parcellating_dbm/outp...,/home/lucagalli/Projects/parcellating_dbm/outp...


## Training the ML Model -- Testing ROC-AUC 

In [42]:
# One vector per subject, preserving model_cohort order
direct_mean_rows = []

for path in model_cohort["direct_mean_path"]:
    values = np.fromfile(path, dtype=np.float64)
    direct_mean_rows.append(values)

# Shape: n_subjects × 83_442
X_direct = np.vstack(direct_mean_rows)

# Labels aligned with the same row order
y = model_cohort["clinical_label_ad"].to_numpy()

print(X_direct.shape)
print(y.shape)


(962, 83442)
(962,)


In [43]:
# One Wasserstein weighted-degree vector per subject,
# preserving the same model_cohort row order
weighted_degree_rows = []

for path in model_cohort["weighted_degree_path"]:
    values = np.fromfile(path, dtype=np.float64)
    weighted_degree_rows.append(values)

# Shape: n_subjects × 83_442
X_weighted_degree = np.vstack(weighted_degree_rows)

# Same labels, aligned with the same subject order
y = model_cohort["clinical_label_ad"].to_numpy()

print(X_weighted_degree.shape)
print(y.shape)

(962, 83442)
(962,)


In [44]:
X_combined = np.hstack([X_direct, X_weighted_degree])

print(X_combined.shape)

(962, 166884)


In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

def evaluate_auc(X, y):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic", LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            C=0.01,
            class_weight="balanced",
            max_iter=5000,
            random_state=42,
        )),
    ])

    # Every score comes from a fold where that subject was not used for training
    oof_probability = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=1,
    )[:, 1]

    return roc_auc_score(y, oof_probability), oof_probability

auc_direct, pred_direct = evaluate_auc(X_direct, y)
auc_weighted, pred_weighted = evaluate_auc(X_weighted_degree, y)
auc_combined, pred_combined = evaluate_auc(X_combined, y)

print(f"Direct Jacobian:  {auc_direct:.3f}")
print(f"Weighted degree:  {auc_weighted:.3f}")
print(f"Combined:         {auc_combined:.3f}")